In [6]:
from typing import Dict, Any, List, Callable

In [13]:
# --- Step 1: Define the Shared Application State Schema ---
# This dictionary represents our tracking tray moving through the assembly line.
class AgentState:
    def __init__(self, user_query: str):
        self.state_data: Dict[str, Any] = {
            "query": user_query,
            "routing_intent": "UNKNOWN",
            "execution_output": "",
            "steps_taken": []  # Maintaining a chronological list of states crossed
        }

    def update(self, key: str, value: Any):
        # PITFALL WARNING FIXED: Handle the list tracking distinctly from direct value assignment
        if key == "steps_taken":
            self.state_data["steps_taken"].append(value)
        else:
            self.state_data[key] = value

In [14]:
# --- Step 2: Define the Graph Nodes (The Workstations) ---

def intent_classifier_node(state: AgentState):
    """Analyzes incoming requests and categorizes downstream routing paths."""
    print("[NODE] Executing Intent Classifier...")
    query = state.state_data["query"].lower()
    
    # Simulating LLM analytical categorization behavior
    if "hardware" in query or "patch" in query:
        intent = "HARDWARE_TRIAGE"
    else:
        intent = "GENERAL_SUPPORT"
        
    state.update("routing_intent", intent)
    state.update("steps_taken", "Classified Intent")

def hardware_execution_node(state: AgentState):
    """Executes high-privilege automated hardware diagnostic logic."""
    print("[NODE] Executing Hardware Triage Protocol...")
    # Simulated execution response
    state.update("execution_output", "DIAGNOSTIC STATUS: Fan array modified to high-performance mode.")
    state.update("steps_taken", "Executed Hardware Fix")

def general_support_node(state: AgentState):
    """Handles basic conversational or informative queries."""
    print("[NODE] Executing General Informational Assistant...")
    state.update("execution_output", "SUPPORT RESPONSE: For billing questions, please review your account portal.")
    state.update("steps_taken", "Generated Standard Answer")

In [15]:
# --- Step 3: Define Conditional Edge Routing Rules ---

def route_based_on_intent(state: AgentState) -> str:
    """Reads the state data and returns the exact string name of the next destination node."""
    intent = state.state_data["routing_intent"]
    print(f"[CONDITIONAL EDGE] Evaluating path routing for intent status: '{intent}'")
    
    if intent == "HARDWARE_TRIAGE":
        return "hardware_node"
    return "general_node"

In [16]:
# --- Step 4: The State Machine Graph Orchestrator Container ---

class MiniLangGraphEngine:
    def __init__(self):
        self.nodes: Dict[str, Callable] = {}
        self.conditional_edge: Callable = None

    def add_node(self, name: str, func: Callable):
        self.nodes[name] = func

    def set_conditional_edge(self, routing_function: Callable):
        self.conditional_edge = routing_function

    def compile_and_run(self, initial_state: AgentState):
        print("--- Compiling Graph Execution Blueprint ---")
        
        # Step 1: Execute the entry point node
        self.nodes["intent_node"](initial_state)
        
        # Step 2: Evaluate the conditional routing edge
        next_destination = self.conditional_edge(initial_state)
        
        # Step 3: Route to the calculated target node
        self.nodes[next_destination](initial_state)
        
        # Step 4: Final verification read out
        print("\n" + "="*60)
        print("GRAPH RUN COMPLETION METADATA AUDIT:")
        print("="*60)
        print(f"Final Execution State Output: {initial_state.state_data['execution_output']}")
        print(f"Audit Trail Steps Logged:    {initial_state.state_data['steps_taken']}")

In [17]:
# --- Step 5: Build and Run Our Pipeline Configuration ---

# Instantiate engine
graph_runtime = MiniLangGraphEngine()

# Register our software processing blocks
graph_runtime.add_node("intent_node", intent_classifier_node)
graph_runtime.add_node("hardware_node", hardware_execution_node)
graph_runtime.add_node("general_node", general_support_node)

# Inject the structural edge split rule
graph_runtime.set_conditional_edge(route_based_on_intent)

# Run Case 1: Triggering the Hardware branch path
state_case_1 = AgentState(user_query="My system is throttling, execute a high-performance fan hardware patch.")
graph_runtime.compile_and_run(state_case_1)

--- Compiling Graph Execution Blueprint ---
[NODE] Executing Intent Classifier...
[CONDITIONAL EDGE] Evaluating path routing for intent status: 'HARDWARE_TRIAGE'
[NODE] Executing Hardware Triage Protocol...

GRAPH RUN COMPLETION METADATA AUDIT:
Final Execution State Output: DIAGNOSTIC STATUS: Fan array modified to high-performance mode.
Audit Trail Steps Logged:    ['Classified Intent', 'Executed Hardware Fix']
